# 📖 Notebook 2: Portfolio & Position Tracking

When a user buys or sells stock, their **portfolio** needs to be updated in real time.
This notebook covers how a brokerage tracks positions, calculates profit/loss, and
keeps everything consistent.

## Learning Objectives

By the end of this notebook you will understand:
- How **positions** track what a user owns
- How trades update positions and cash balances
- How to calculate **unrealized P&L** (profit/loss on open positions)
- How to calculate **realized P&L** (profit/loss from completed sales)
- Why we use a **positions table** instead of recalculating from trades every time

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/robinhood
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM positions")
    print(f"✅ PostgreSQL connected — {cur.fetchone()[0]} positions loaded")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 📚 What Is a Position?

A **position** represents how many shares of a stock a user currently holds,
and at what average cost they bought them.

```
  Alice's Portfolio
  ┌────────┬──────────┬──────────────┬──────────────┐
  │ Symbol │ Quantity │   Avg Cost   │ Market Value │
  ├────────┼──────────┼──────────────┼──────────────┤
  │ AAPL   │    50    │   $180.00    │   $191.50    │
  │ META   │    20    │   $500.00    │   $522.10    │
  └────────┴──────────┴──────────────┴──────────────┘
                        ↑ what she paid   ↑ current price
```

### Why a Positions Table?

We *could* calculate positions by scanning all trades every time, but:
- A user with 10,000 trades would need 10,000 rows scanned for every portfolio view
- This gets slower as the user trades more

Instead, we keep a **materialized position** — one row per (user, symbol) — and
update it whenever a trade executes. This is an O(1) lookup vs O(n) scan.

In [ ]:
# Let's look at Alice's current positions (from our seed data)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT s.ticker, p.quantity, 
           p.avg_cost_cents / 100.0 AS avg_cost,
           s.last_price_cents / 100.0 AS current_price,
           (s.last_price_cents - p.avg_cost_cents) * p.quantity / 100.0 AS unrealized_pnl
    FROM positions p
    JOIN symbols s ON p.symbol_id = s.id
    WHERE p.user_id = 1
    ORDER BY s.ticker
""")

positions = cur.fetchall()

print("📊 Alice's Portfolio")
print("=" * 70)
print(f"{'Ticker':<8} {'Qty':>5} {'Avg Cost':>10} {'Mkt Price':>10} {'P&L':>12}")
print("-" * 70)

total_value = 0
total_cost = 0
for pos in positions:
    pnl = float(pos['unrealized_pnl'])
    pnl_str = f"{'🟢' if pnl >= 0 else '🔴'} ${abs(pnl):,.2f}"
    print(f"{pos['ticker']:<8} {pos['quantity']:>5} ${float(pos['avg_cost']):>9,.2f} "
          f"${float(pos['current_price']):>9,.2f} {pnl_str:>12}")
    total_value += float(pos['current_price']) * pos['quantity']
    total_cost += float(pos['avg_cost']) * pos['quantity']

print("-" * 70)
total_pnl = total_value - total_cost
print(f"{'Total':<8} {'':>5} ${total_cost:>9,.2f} ${total_value:>9,.2f} "
      f"{'🟢' if total_pnl >= 0 else '🔴'} ${abs(total_pnl):>9,.2f}")

conn.close()

## Bad Practice -> Best Practice: Scan Trades vs Materialized Positions

A new engineer might say *"we already store every trade -- why keep a `positions` table at all? Just `SUM()` the trades when the user opens their portfolio."*

That works for 10 trades. It stops working at 10,000. Let's measure the difference.

- **Bad (O(n))**: scan every trade the user ever made and re-aggregate. Gets slower forever as the user trades more.
- **Good (O(1))**: keep one row per `(user, symbol)` in `positions`, updated inside the same DB transaction as the trade. Constant-time lookup no matter how much history.


In [ ]:
import time as _time
import random

# Seeded: the synthetic trade history must be the same on every run,
# otherwise the timing numbers below wander for no reason.
random.seed(20260821)

# Build a synthetic user (#10) with a few thousand trades so we can actually feel the difference.
conn = get_db()
cur = conn.cursor()

cur.execute("DELETE FROM trades WHERE order_id IN (SELECT id FROM orders WHERE user_id = 10)")
cur.execute("DELETE FROM orders WHERE user_id = 10")
cur.execute("DELETE FROM positions WHERE user_id = 10")

N = 5000
cur.execute("SELECT id, last_price_cents FROM symbols ORDER BY id")
symbols_list = cur.fetchall()   # [(id, price_cents), ...]

# Insert N filled market-buy orders in one batch
order_rows = []
for _ in range(N):
    sid, base = random.choice(symbols_list)
    price = base + random.randint(-200, 200)
    qty = random.randint(1, 5)
    order_rows.append((10, sid, "buy", "market", qty, "filled", qty, price))

cur.executemany("""
    INSERT INTO orders (user_id, symbol_id, side, order_type, quantity,
                        status, filled_quantity, filled_avg_price_cents)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
""", order_rows)

# Create one trade per order
cur.execute("""
    SELECT id, symbol_id, filled_quantity, filled_avg_price_cents
    FROM orders WHERE user_id = 10
""")
trade_rows = [(oid, sid, price, qty) for oid, sid, qty, price in cur.fetchall()]
cur.executemany("INSERT INTO trades (order_id, symbol_id, price_cents, quantity) VALUES (%s, %s, %s, %s)",
                trade_rows)

print(f"Seeded user #10 with {N} trades across {len(symbols_list)} symbols.")


# BAD approach: recompute positions from the trades table every time
def bad_positions_for_user(user_id):
    cur.execute("""
        SELECT s.ticker,
               SUM(CASE WHEN o.side='buy'  THEN t.quantity ELSE -t.quantity END) AS qty,
               SUM(CASE WHEN o.side='buy'  THEN t.quantity * t.price_cents ELSE 0 END)::float
                 / NULLIF(SUM(CASE WHEN o.side='buy' THEN t.quantity ELSE 0 END), 0) AS avg_cost_cents
        FROM trades t
        JOIN orders o  ON o.id = t.order_id
        JOIN symbols s ON s.id = t.symbol_id
        WHERE o.user_id = %s
        GROUP BY s.ticker
        HAVING SUM(CASE WHEN o.side='buy' THEN t.quantity ELSE -t.quantity END) > 0
    """, (user_id,))
    return cur.fetchall()


# GOOD approach: read straight from the materialized positions table.
# First we build it once -- normally this happens trade-by-trade inside a transaction.
cur.execute("""
    INSERT INTO positions (user_id, symbol_id, quantity, avg_cost_cents)
    SELECT o.user_id,
           t.symbol_id,
           SUM(t.quantity),
           (SUM(t.quantity * t.price_cents) / SUM(t.quantity))::bigint
    FROM trades t JOIN orders o ON o.id = t.order_id
    WHERE o.user_id = 10 AND o.side = 'buy'
    GROUP BY o.user_id, t.symbol_id
""")

def good_positions_for_user(user_id):
    cur.execute("""
        SELECT s.ticker, p.quantity, p.avg_cost_cents
        FROM positions p JOIN symbols s ON s.id = p.symbol_id
        WHERE p.user_id = %s
    """, (user_id,))
    return cur.fetchall()


# Time both approaches (average over a few runs so we don't measure cache warm-up)
def time_it(fn, repeats=5):
    start = _time.perf_counter()
    for _ in range(repeats):
        fn(10)
    return (_time.perf_counter() - start) * 1000 / repeats

# Warm the cache so both approaches get a fair shot
bad_positions_for_user(10); good_positions_for_user(10)

bad_ms  = time_it(bad_positions_for_user)
good_ms = time_it(good_positions_for_user)

print(f"BAD  (scan {N} trades):        {bad_ms:8.2f} ms / call")
print(f"GOOD (read positions table):  {good_ms:8.2f} ms / call")
assert good_ms > 0, "timer resolution too coarse to compare"
print(f"Speedup: ~{bad_ms / good_ms:.1f}x")
print()

# If this ever stops holding, the lab is no longer demonstrating its own lesson:
# scanning 5,000 trade rows must be measurably slower than one indexed lookup.
assert bad_ms > good_ms * 1.5, (
    f"expected scanning {N} trades to be much slower than the positions lookup, "
    f"got bad={bad_ms:.2f} ms vs good={good_ms:.2f} ms"
)

# ...and both must return the SAME answer. A fast query that disagrees with the
# ledger is worse than a slow one.
scanned = {t: (int(q), round(float(a))) for t, q, a in bad_positions_for_user(10)}
stored = {t: (int(q), int(a)) for t, q, a in good_positions_for_user(10)}
assert scanned.keys() == stored.keys(), (scanned.keys(), stored.keys())
for ticker, (qty, avg) in scanned.items():
    s_qty, s_avg = stored[ticker]
    assert qty == s_qty, f"{ticker}: trades say {qty} shares, positions table says {s_qty}"
    assert abs(avg - s_avg) <= 1, f"{ticker}: avg cost {avg} vs {s_avg}"
print(f"✅ materialized positions match a full rescan of all {N} trades "
      f"({len(stored)} symbols)")
print()

print("Tip: the 'bad' query gets slower forever as the user keeps trading.")
print("     The 'good' query is the same cost whether the user has 5 trades or 5 million.")

# Clean up the synthetic user right away so the rest of the notebook sees clean state
cur.execute("DELETE FROM positions WHERE user_id = 10")
cur.execute("DELETE FROM trades WHERE order_id IN (SELECT id FROM orders WHERE user_id = 10)")
cur.execute("DELETE FROM orders WHERE user_id = 10")
conn.close()


## 🔄 Updating Positions When Trades Happen

When a BUY trade executes, we need to:
1. Write the `orders` row and the `trades` row — the ledger entry
2. Increase the position quantity
3. Update the average cost
4. Deduct cash from the user's balance

When a SELL trade executes, we need to:
1. Write the `orders` row and the `trades` row
2. Decrease the position quantity
3. Keep average cost the same (it's the cost of shares we still hold)
4. Add cash to the user's balance
5. Calculate realized P&L

All of it lands in **one transaction**. A `trades` row with no position update
means the portfolio silently drifts from the ledger. A position update with no
`trades` row is worse: the portfolio can never be rebuilt from history, and no
audit will ever reconcile.

### Average Cost Calculation

When buying more shares of something you already own:

```
new_avg_cost = (old_qty × old_avg_cost + new_qty × new_price) / (old_qty + new_qty)
```

Example: Alice has 50 AAPL at $180. She buys 10 more at $191.50:
```
new_avg_cost = (50 × $180 + 10 × $191.50) / (50 + 10) = $181.92
```

$181.91666… is not a whole number of cents, so we round half-up **in integer
arithmetic** — no floats anywhere near money (Notebook 1 showed why). Rounding
does leave a few cents of cost basis unaccounted for. A production ledger dodges
the drift entirely by storing the *exact* total cost basis in cents and deriving
the average only for display; we keep the single `avg_cost_cents` column here to
stay close to the schema in `db/init.sql`.


In [ ]:
def process_trade(user_id: int, ticker: str, side: str,
                  quantity: int, price_cents: int):
    """
    Process a filled trade: write the order + trade rows, update the position
    and update the cash balance — all inside a single transaction.

    Returns the new order id, or None if the trade was rejected.
    """
    conn = psycopg2.connect(**DB_CONFIG)
    # Use a transaction (no autocommit) for atomicity
    conn.autocommit = False
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Look up symbol
        cur.execute("SELECT id FROM symbols WHERE ticker = %s", (ticker,))
        symbol = cur.fetchone()
        symbol_id = symbol['id']

        total_cost_cents = price_cents * quantity

        # --- The ledger entry. This is not optional bookkeeping: the orders +
        # --- trades rows ARE the record that the money moved. Everything below
        # --- (position, balance) is a materialized view of them.
        cur.execute("""
            INSERT INTO orders (user_id, symbol_id, side, order_type, quantity,
                                status, filled_quantity, filled_avg_price_cents)
            VALUES (%s, %s, %s, 'market', %s, 'filled', %s, %s)
            RETURNING id
        """, (user_id, symbol_id, side, quantity, quantity, price_cents))
        order_id = cur.fetchone()['id']

        cur.execute("""
            INSERT INTO trades (order_id, symbol_id, price_cents, quantity)
            VALUES (%s, %s, %s, %s)
        """, (order_id, symbol_id, price_cents, quantity))

        if side == 'buy':
            # Deduct cash
            cur.execute("""
                UPDATE users SET balance_cents = balance_cents - %s
                WHERE id = %s AND balance_cents >= %s
                RETURNING balance_cents
            """, (total_cost_cents, user_id, total_cost_cents))
            result = cur.fetchone()
            if not result:
                raise ValueError("Insufficient funds!")

            # Update or create position
            cur.execute("""
                SELECT quantity, avg_cost_cents FROM positions
                WHERE user_id = %s AND symbol_id = %s
            """, (user_id, symbol_id))
            existing = cur.fetchone()

            if existing:
                old_qty = existing['quantity']
                old_cost = existing['avg_cost_cents']
                new_qty = old_qty + quantity
                # Weighted average cost, rounded half-up with integer maths only
                total_basis = old_qty * old_cost + quantity * price_cents
                new_avg = (total_basis + new_qty // 2) // new_qty
                cur.execute("""
                    UPDATE positions SET quantity = %s, avg_cost_cents = %s, updated_at = NOW()
                    WHERE user_id = %s AND symbol_id = %s
                """, (new_qty, new_avg, user_id, symbol_id))
                print(f"📈 Updated position: {old_qty} → {new_qty} shares, "
                      f"avg cost ${old_cost/100:.2f} → ${new_avg/100:.2f}")
            else:
                cur.execute("""
                    INSERT INTO positions (user_id, symbol_id, quantity, avg_cost_cents)
                    VALUES (%s, %s, %s, %s)
                """, (user_id, symbol_id, quantity, price_cents))
                print(f"📈 New position: {quantity} shares @ ${price_cents/100:.2f}")

            print(f"💰 Cash deducted: ${total_cost_cents/100:,.2f}  "
                  f"(balance: ${result['balance_cents']/100:,.2f})")

        else:  # sell
            # Check position exists and has enough shares
            cur.execute("""
                SELECT quantity, avg_cost_cents FROM positions
                WHERE user_id = %s AND symbol_id = %s
            """, (user_id, symbol_id))
            existing = cur.fetchone()

            if not existing or existing['quantity'] < quantity:
                raise ValueError(f"Not enough shares! Have {existing['quantity'] if existing else 0}, "
                                 f"trying to sell {quantity}")

            old_qty = existing['quantity']
            avg_cost = existing['avg_cost_cents']
            new_qty = old_qty - quantity

            # Calculate realized P&L
            realized_pnl_cents = (price_cents - avg_cost) * quantity

            if new_qty == 0:
                cur.execute("""
                    DELETE FROM positions WHERE user_id = %s AND symbol_id = %s
                """, (user_id, symbol_id))
            else:
                cur.execute("""
                    UPDATE positions SET quantity = %s, updated_at = NOW()
                    WHERE user_id = %s AND symbol_id = %s
                """, (new_qty, user_id, symbol_id))

            # Add cash
            cur.execute("""
                UPDATE users SET balance_cents = balance_cents + %s
                WHERE id = %s RETURNING balance_cents
            """, (total_cost_cents, user_id))
            result = cur.fetchone()

            pnl_emoji = '🟢' if realized_pnl_cents >= 0 else '🔴'
            print(f"📉 Sold {quantity} shares: {old_qty} → {new_qty} shares")
            print(f"{pnl_emoji} Realized P&L: ${realized_pnl_cents/100:,.2f}")
            print(f"💰 Cash added: ${total_cost_cents/100:,.2f}  "
                  f"(balance: ${result['balance_cents']/100:,.2f})")

        conn.commit()
        print(f"✅ Transaction committed (order #{order_id}, trade recorded)")

        # The portfolio cache is now stale — drop it. (Only after the commit:
        # invalidating before it would race a reader into caching old rows.)
        get_redis().delete(f"portfolio:{user_id}")
        return order_id

    except Exception as e:
        conn.rollback()
        print(f"❌ Transaction rolled back: {e}")
        return None
    finally:
        conn.close()


In [ ]:
# Alice buys 10 more AAPL at $191.50

print("Alice buys 10 more AAPL at $191.50")
print("=" * 50)
alice_buy_id = process_trade(user_id=1, ticker="AAPL", side="buy", quantity=10, price_cents=19150)


In [ ]:
# Alice sells 20 META at $530.00 (bought at $500)

print("Alice sells 20 META at $530.00")
print("=" * 50)
alice_sell_id = process_trade(user_id=1, ticker="META", side="sell", quantity=20, price_cents=53000)


In [ ]:
# Bob buys a new stock he didn't own before

print("Bob buys 15 AAPL at $191.00")
print("=" * 50)
bob_buy_id = process_trade(user_id=2, ticker="AAPL", side="buy", quantity=15, price_cents=19100)


### ✅ Audit: Do the Ledger, the Positions and the Cash Agree?

Three trades just moved money. In a brokerage the only acceptable answer to
"did that work?" is a reconciliation, so let's do one: every order marked
`filled` must be backed by `trades` rows summing to exactly `filled_quantity`,
and the resulting positions and balances must be the ones the arithmetic above
predicts.

> Re-running the trade cells above without running the **Cleanup** cell at the
> bottom first will (correctly) trip these assertions — the state would no
> longer be "seed data plus these three trades".


In [ ]:
demo_order_ids = [alice_buy_id, alice_sell_id, bob_buy_id]
assert all(oid is not None for oid in demo_order_ids), \
    "a demo trade was rejected — see the ❌ message above"

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# 1. No filled order without a matching ledger entry. This is the defect that
#    hides best: status says 'filled', the trades table says nothing happened.
cur.execute("""
    SELECT o.id, o.filled_quantity, COALESCE(SUM(t.quantity), 0) AS ledger_qty
    FROM orders o
    LEFT JOIN trades t ON t.order_id = o.id
    WHERE o.id = ANY(%s)
    GROUP BY o.id, o.filled_quantity
""", (demo_order_ids,))
ledger = cur.fetchall()
assert len(ledger) == 3, f"expected 3 demo orders, found {len(ledger)}"
for row in ledger:
    assert row['ledger_qty'] == row['filled_quantity'], (
        f"order #{row['id']} says {row['filled_quantity']} shares filled but the "
        f"trades table sums to {row['ledger_qty']}"
    )

# 2. Positions match the weighted-average arithmetic from the markdown above.
def position(user_id, ticker):
    cur.execute("""
        SELECT p.quantity, p.avg_cost_cents
        FROM positions p JOIN symbols s ON s.id = p.symbol_id
        WHERE p.user_id = %s AND s.ticker = %s
    """, (user_id, ticker))
    return cur.fetchone()

alice_aapl = position(1, "AAPL")
assert alice_aapl['quantity'] == 60, alice_aapl          # 50 held + 10 bought
assert alice_aapl['avg_cost_cents'] == 18192, alice_aapl # (50*18000 + 10*19150)/60
assert position(1, "META") is None, "Alice sold her whole META position"

bob_aapl = position(2, "AAPL")
assert bob_aapl['quantity'] == 15 and bob_aapl['avg_cost_cents'] == 19100, bob_aapl

# 3. Cash moved by exactly the traded notional — no more, no less.
cur.execute("SELECT id, balance_cents FROM users WHERE id IN (1, 2) ORDER BY id")
balances = {row['id']: row['balance_cents'] for row in cur.fetchall()}
START = 10_000_000                                        # $100,000.00 seed balance
expected_alice = START - 10 * 19150 + 20 * 53000
expected_bob = START - 15 * 19100
assert balances[1] == expected_alice, (balances[1], expected_alice)
assert balances[2] == expected_bob, (balances[2], expected_bob)

conn.close()

print("✅ Ledger, positions and cash all reconcile:")
print(f"   Alice: 60 AAPL @ $181.92 avg, no META, cash ${expected_alice/100:,.2f}")
print(f"   Bob:   15 AAPL @ $191.00 avg,          cash ${expected_bob/100:,.2f}")
print("   Every filled order is backed by trade rows summing to its fill quantity.")


## 💨 Caching Portfolios with Redis

Users check their portfolio **constantly**. Every time they open the app, every time
a price changes, the portfolio recalculates.

We can cache portfolio data in Redis to avoid hitting Postgres on every view.
The cache is invalidated whenever a trade executes.

In [ ]:
r = get_redis()

def get_portfolio(user_id: int) -> list:
    """Get portfolio with Redis cache-aside pattern."""
    cache_key = f"portfolio:{user_id}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        print(f"⚡ Cache HIT for user {user_id}")
        return json.loads(cached)

    print(f"🐘 Cache MISS for user {user_id} — querying Postgres")

    # Fetch from database
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT s.ticker, p.quantity,
               p.avg_cost_cents, s.last_price_cents
        FROM positions p
        JOIN symbols s ON p.symbol_id = s.id
        WHERE p.user_id = %s
        ORDER BY s.ticker
    """, (user_id,))
    positions = [dict(row) for row in cur.fetchall()]
    conn.close()

    # Store in cache with 30 second TTL
    r.setex(cache_key, 30, json.dumps(positions))

    return positions


def invalidate_portfolio_cache(user_id: int):
    """Called after every trade to ensure fresh data."""
    r.delete(f"portfolio:{user_id}")
    print(f"🗑️  Cache invalidated for user {user_id}")


# First call: cache miss
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")
print()

# Second call: cache hit
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")
print()

# After a trade: invalidate
invalidate_portfolio_cache(1)
print()

# Next call: cache miss again
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")

## 📊 Portfolio Summary: Tying It All Together

Let's build a complete portfolio view that shows:
- Each position with current value
- Unrealized P&L per position
- Total portfolio value
- Cash balance

In [ ]:
def display_portfolio(user_id: int):
    """Display a full portfolio summary for a user."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get user info
    cur.execute("SELECT username, balance_cents FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()

    # Get positions with current prices
    cur.execute("""
        SELECT s.ticker, p.quantity,
               p.avg_cost_cents / 100.0 AS avg_cost,
               s.last_price_cents / 100.0 AS market_price,
               p.quantity * s.last_price_cents / 100.0 AS market_value,
               p.quantity * p.avg_cost_cents / 100.0 AS cost_basis,
               (s.last_price_cents - p.avg_cost_cents) * p.quantity / 100.0 AS unrealized_pnl
        FROM positions p
        JOIN symbols s ON p.symbol_id = s.id
        WHERE p.user_id = %s
        ORDER BY p.quantity * s.last_price_cents DESC
    """, (user_id,))
    positions = cur.fetchall()
    conn.close()

    cash = user['balance_cents'] / 100
    total_market_value = sum(float(p['market_value']) for p in positions)
    total_cost = sum(float(p['cost_basis']) for p in positions)
    total_pnl = total_market_value - total_cost
    total_portfolio = total_market_value + cash

    print(f"\n{'='*70}")
    print(f"  📊 {user['username'].upper()}'s Portfolio")
    print(f"{'='*70}")
    print(f"{'Ticker':<8} {'Qty':>5} {'Avg Cost':>10} {'Price':>10} {'Value':>12} {'P&L':>12}")
    print("-" * 70)

    for pos in positions:
        pnl = float(pos['unrealized_pnl'])
        emoji = '🟢' if pnl >= 0 else '🔴'
        print(f"{pos['ticker']:<8} {pos['quantity']:>5} "
              f"${float(pos['avg_cost']):>9,.2f} ${float(pos['market_price']):>9,.2f} "
              f"${float(pos['market_value']):>10,.2f} {emoji}${abs(pnl):>9,.2f}")

    print("-" * 70)
    pnl_emoji = '🟢' if total_pnl >= 0 else '🔴'
    print(f"{'Stocks':<8} {'':>5} {'':>10} {'':>10} ${total_market_value:>10,.2f} "
          f"{pnl_emoji}${abs(total_pnl):>9,.2f}")
    print(f"{'Cash':<8} {'':>5} {'':>10} {'':>10} ${cash:>10,.2f}")
    print(f"{'─'*70}")
    print(f"{'TOTAL':<8} {'':>5} {'':>10} {'':>10} ${total_portfolio:>10,.2f}")
    print()


# View portfolios for Alice and Bob
display_portfolio(1)  # Alice
display_portfolio(2)  # Bob

## 🧹 Cleanup

In [ ]:
# Reset positions, balances and the demo ledger to seed-data state
conn = get_db()
cur = conn.cursor()

# Drop the orders + trades this notebook wrote (seed data is order ids 1-5).
# Trades first: they hold a foreign key to orders.
cur.execute("""
    DELETE FROM trades WHERE order_id IN (
        SELECT id FROM orders WHERE user_id IN (1, 2) AND id > 5
    )
""")
cur.execute("DELETE FROM orders WHERE user_id IN (1, 2) AND id > 5")

# Reset Alice: 50 AAPL @ 180, 20 META @ 500
cur.execute("DELETE FROM positions WHERE user_id = 1")
cur.execute("""
    INSERT INTO positions (user_id, symbol_id, quantity, avg_cost_cents) VALUES
    (1, 1, 50, 18000), (1, 3, 20, 50000)
""")
cur.execute("UPDATE users SET balance_cents = 10000000 WHERE id IN (1, 2)")

# Reset Bob: 10 NVDA @ 850
cur.execute("DELETE FROM positions WHERE user_id = 2 AND symbol_id != 7")
cur.execute("UPDATE positions SET quantity = 10, avg_cost_cents = 85000 WHERE user_id = 2 AND symbol_id = 7")

# Clear Redis cache
r = get_redis()
for key in r.keys("portfolio:*"):
    r.delete(key)

print("🧹 Reset positions, balances, ledger and cache to seed-data state")
conn.close()


## 📚 Summary

### Key Takeaways

1. **Positions table** stores one row per (user, symbol) for O(1) portfolio lookups.
   (See the timing demo: scanning trades is O(n) and only gets worse.)
2. **The `trades` rows are the ledger; positions are a materialized view of them.**
   The audit cell proves the two still agree after every trade.
3. **Average cost** is recalculated on each buy using a weighted average, rounded
   half-up in integer cents. Production ledgers store the exact total cost basis
   instead, to avoid the sub-cent drift that rounding leaves behind.
4. **Order, trade, position and balance are written in ONE transaction.** An order
   marked `filled` with no `trades` row behind it is the classic silent money bug.
5. **Unrealized P&L** = (current price − avg cost) × quantity (paper gains/losses).
6. **Realized P&L** = (sell price − avg cost) × quantity (actual gains/losses when you sell).
7. **Cache portfolios in Redis** with cache-aside; `process_trade` invalidates the key
   *after* the commit, so no reader can cache pre-trade rows.

### For System Design Interviews

- Explain why you keep a materialized positions table (performance)
- Mention ACID transactions for balance + position updates, and that the trade row
  goes in the same transaction
- Discuss cache-aside for frequently-viewed portfolio data
- Note that positions are partitioned by `user_id` for single-node queries

### What This Toy Does NOT Do

- **No short positions** — quantity can only go to zero, never negative.
- **No lot-level cost basis** (FIFO/LIFO/specific-ID), which is what tax reporting
  actually needs; we keep one blended average per symbol.
- **No settlement, no T+1, no buying power** — cash moves instantly and there is no
  distinction between settled and unsettled funds.
- **No corporate actions** — splits, dividends and mergers all rewrite cost basis.
- **Realized P&L is printed, never stored.** A real system persists it per lot.

### Next Up

In **Notebook 3**, we'll build **real-time market data streaming** — using Kafka
for trade feeds and Redis pub/sub to push live prices to users.
